# Research strategies from a notebook

An idea expressed as a `ResearchStrategy` subclass can be run end to end in one cell: the
orchestrator pulls the exchange data when missing, computes the features, runs the .NET
engine, and hands back a `RunResult` (rows + manifest + plain-text summary) that pandas can
consume straight away.

Prereqs: the editable `quantlab` install (`pip install -e Research/Python`) and the built
runner (`dotnet build Research\Runner\QuantConnect.Research.Runner.csproj -c Release`).

In [ ]:
from quantlab.research import ResearchStrategy, load_script

Write the strategy to a file (‘’%%writefile’’) so the engine can import it by path — the
declaration lives entirely in Python:

- `exchange`/`symbols`/`event_types` say what to load (pulled on first run when missing)
- `features` are computed and passed to `on_observation(observation, features)`
- your `print()` calls land in `strategy.log`, not the (invisible) engine stdout

In [ ]:
%%writefile sma_quick.py
from quantlab.research import ResearchStrategy


class SmaQuick(ResearchStrategy):
    exchange = "binance"
    symbols = ["BTCUSDT"]
    event_types = ["Trade", "Quote"]
    interval_seconds = 60
    start = "2024-01-01T00:00:00"
    end = "2024-01-01T00:40:00"
    features = ["mid_price", "trade_flow"]

    def initialize(self, context):
        print("SmaQuick on", context.get("symbols"))

    def on_observation(self, observation, features):
        flow = features.get("trade_flow", 0.0) or 0.0
        if abs(flow) > 0:
            return {"action": "buy" if flow > 0 else "sell", "qty": 1}
        return None

    def finalize(self):
        return {"outcome": "done"}

`run()` returns a `RunResult`. Pass `sample_seconds` to replay only the most recent
that-many seconds of the window (great for iterating on an idea); leave `start`/`end`
empty to have the window derived from the staged data.

In [ ]:
module = load_script("sma_quick.py")
cls = next(cls for cls in vars(module).values()
           if isinstance(cls, type) and issubclass(cls, ResearchStrategy) and cls is not ResearchStrategy)

result = cls.run(sample_seconds=900)
print(result.summary)

In [ ]:
result.report_df().head()

In [ ]:
result.signals_df()

In [ ]:
# percentage of observations with a signal, from the manifest-backed metrics
obs = result.metrics.get("observation_count", 0)
sig = result.metrics.get("signals", 0)
(sig / obs if obs else 0.0)

The strategy’s own output was captured to `results/<job>/strategy.log` (the engine’s
embedded interpreter has no visible stdout). `RunResult` also carries `manifest`, `exit_code`
and the flat CSV routes under `results_dir`.

Typo a hook (say `on_observaton`) and you get a `did you mean 'on_observation'?` error
instead of a silent empty run.

In [ ]:
from pathlib import Path

log = Path(result.results_dir) / "strategy.log"
print(log.read_text()[:400])